# Notebook 04 – Demand Forecasting Model

## Objective

This notebook builds the demand forecasting model for Project FORESIGHT.

The implementation follows the client engagement brief and satisfies Deliverable D3.

### Goals

- Build a weekly SKU-level forecasting dataset
- Create a Seasonal Naive baseline
- Engineer lag and rolling features without leakage
- Train a Random Forest model
- Compare against the baseline using WAPE
- Evaluate with Rolling-Origin Cross Validation

In [1]:
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

In [2]:
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"

PROCESSED_DATA_DIR = DATA_DIR / "processed"

FEATURE_DIR = PROCESSED_DATA_DIR / "feature_chunks"

## Load Feature Engineered Data

The feature-engineered chunks created in Notebook 03 are loaded for building
the weekly forecasting dataset.

Working with chunked data keeps the pipeline memory-efficient and reproducible,
allowing the notebook to process the full dataset without exceeding RAM limits.

In [3]:
# ============================================================
# Load Feature Chunks
# ============================================================

feature_files = sorted(
    FEATURE_DIR.glob("final_chunk_*.parquet")
)

print(f"Feature Chunks Found : {len(feature_files)}")

Feature Chunks Found : 64


In [4]:
# ============================================================
# Inspect Sample Chunk
# ============================================================

sample = pd.read_parquet(feature_files[0])

print("Shape :", sample.shape)

sample.head()

Shape : (914700, 34)


,id,item_id,dept_id,cat_id,store_id,state_id,d,units_sold,date,wm_yr_wk,...,quarter,day_of_year,week_of_year,is_month_start,is_month_end,is_weekend,has_price,log_price,has_event,has_snap
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,29,4,0,0,1,0,0.0,0,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,29,4,0,0,1,0,0.0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,29,4,0,0,1,0,0.0,0,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,29,4,0,0,1,0,0.0,0,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,29,4,0,0,1,0,0.0,0,0


## Create Weekly Modeling Dataset

The engagement brief requires a **weekly SKU-level forecast** rather than a
daily forecast.

Therefore, the daily observations are aggregated into weekly demand records
for each SKU and store.

Weekly aggregation:

- Reduces dataset size
- Matches the project scope
- Improves model stability
- Makes lag features meaningful

In [5]:
for f in FEATURE_DIR.glob("*.parquet"):
    print(f.name)

final_chunk_001.parquet
final_chunk_002.parquet
final_chunk_003.parquet
final_chunk_004.parquet
final_chunk_005.parquet
final_chunk_006.parquet
final_chunk_007.parquet
final_chunk_008.parquet
final_chunk_009.parquet
final_chunk_010.parquet
final_chunk_011.parquet
final_chunk_012.parquet
final_chunk_013.parquet
final_chunk_014.parquet
final_chunk_015.parquet
final_chunk_016.parquet
final_chunk_017.parquet
final_chunk_018.parquet
final_chunk_019.parquet
final_chunk_020.parquet
final_chunk_021.parquet
final_chunk_022.parquet
final_chunk_023.parquet
final_chunk_024.parquet
final_chunk_025.parquet
final_chunk_026.parquet
final_chunk_027.parquet
final_chunk_028.parquet
final_chunk_029.parquet
final_chunk_030.parquet
final_chunk_031.parquet
final_chunk_032.parquet
final_chunk_033.parquet
final_chunk_034.parquet
final_chunk_035.parquet
final_chunk_036.parquet
final_chunk_037.parquet
final_chunk_038.parquet
final_chunk_039.parquet
final_chunk_040.parquet
final_chunk_041.parquet
final_chunk_042.

In [6]:
# ============================================================
# Locate Feature Chunks
# ============================================================

FEATURE_DIR = PROCESSED_DATA_DIR / "feature_chunks"

feature_chunk_files = sorted(
    FEATURE_DIR.glob("final_chunk_*.parquet")
)

print(f"Feature Chunks Found: {len(feature_chunk_files)}")

if len(feature_chunk_files) > 0:
    print("First chunk:", feature_chunk_files[0].name)
else:
    print("❌ No feature chunks found.")

Feature Chunks Found: 64
First chunk: final_chunk_001.parquet


In [7]:
# ============================================================
# Create Weekly Dataset Chunk-by-Chunk
# ============================================================

import gc

WEEKLY_DIR = PROCESSED_DATA_DIR / "weekly_chunks"
WEEKLY_DIR.mkdir(exist_ok=True)

for i, file in enumerate(feature_chunk_files, start=1):

    print(f"Processing chunk {i}/{len(feature_chunk_files)}")

    chunk = pd.read_parquet(file)

    chunk["date"] = pd.to_datetime(chunk["date"])

    weekly = (
        chunk
        .groupby(
            [
                "item_id",
                "store_id",
                "year",
                "week_of_year"
            ],
            as_index=False
        )
        .agg(
            units_sold=("units_sold", "sum"),
            sell_price=("sell_price", "mean"),
            has_event=("has_event", "max"),
            has_snap=("has_snap", "max"),
            is_weekend=("is_weekend", "max")
        )
    )

    weekly.to_parquet(
        WEEKLY_DIR / f"weekly_chunk_{i:03d}.parquet",
        index=False
    )

    del chunk
    del weekly
    gc.collect()

print("\n✅ Weekly datasets created successfully.")

Processing chunk 1/64
Processing chunk 2/64
Processing chunk 3/64
Processing chunk 4/64
Processing chunk 5/64
Processing chunk 6/64
Processing chunk 7/64
Processing chunk 8/64
Processing chunk 9/64
Processing chunk 10/64
Processing chunk 11/64
Processing chunk 12/64
Processing chunk 13/64
Processing chunk 14/64
Processing chunk 15/64
Processing chunk 16/64
Processing chunk 17/64
Processing chunk 18/64
Processing chunk 19/64
Processing chunk 20/64
Processing chunk 21/64
Processing chunk 22/64
Processing chunk 23/64
Processing chunk 24/64
Processing chunk 25/64
Processing chunk 26/64
Processing chunk 27/64
Processing chunk 28/64
Processing chunk 29/64
Processing chunk 30/64
Processing chunk 31/64
Processing chunk 32/64
Processing chunk 33/64
Processing chunk 34/64
Processing chunk 35/64
Processing chunk 36/64
Processing chunk 37/64
Processing chunk 38/64
Processing chunk 39/64
Processing chunk 40/64
Processing chunk 41/64
Processing chunk 42/64
Processing chunk 43/64
Processing chunk 44/

In [8]:
weekly_chunk_files = sorted(
    WEEKLY_DIR.glob("weekly_chunk_*.parquet")
)

print("Weekly Chunks:", len(weekly_chunk_files))

sample = pd.read_parquet(weekly_chunk_files[0])

print(sample.shape)

sample.head()

Weekly Chunks: 64
(152450, 9)


,item_id,store_id,year,week_of_year,units_sold,sell_price,has_event,has_snap,is_weekend
0,FOODS_1_001,CA_1,2011,4,3,2.0,0,0,1
1,FOODS_1_001,CA_1,2011,5,9,2.0,1,1,1
2,FOODS_1_001,CA_1,2011,6,7,2.0,0,1,1
3,FOODS_1_001,CA_1,2011,7,10,2.0,1,1,1
4,FOODS_1_001,CA_1,2011,8,14,2.0,1,0,1


In [9]:
import pandas as pd
import gc

weekly_frames = []

for file in weekly_chunk_files:
    weekly_frames.append(pd.read_parquet(file))

weekly_df = pd.concat(weekly_frames, ignore_index=True)

print("Shape:", weekly_df.shape)

print(
    "Memory Usage:",
    round(
        weekly_df.memory_usage(deep=True).sum() / 1024**2,
        2
    ),
    "MB"
)

Shape: (10153170, 9)
Memory Usage: 539.76 MB


## Create Sequential Time Index

Why?

Time-series forecasting requires observations to be ordered chronologically.

Instead of relying only on calendar week numbers, a sequential index is created for every SKU-store combination. This index preserves temporal order and enables lag and rolling-window feature engineering without introducing future information

In [10]:
# ============================================================
# Create Time Index
# ============================================================

weekly_df = weekly_df.sort_values(
    [
        "item_id",
        "store_id",
        "year",
        "week_of_year"
    ]
).reset_index(drop=True)

weekly_df["time_idx"] = (
    weekly_df
    .groupby(["item_id", "store_id"])
    .cumcount()
)

weekly_df.head()

,item_id,store_id,year,week_of_year,units_sold,sell_price,has_event,has_snap,is_weekend,time_idx
0,FOODS_1_001,CA_1,2011,4,3,2.0,0,0,1,0
1,FOODS_1_001,CA_1,2011,5,9,2.0,1,1,1,1
2,FOODS_1_001,CA_1,2011,6,7,2.0,0,1,1,2
3,FOODS_1_001,CA_1,2011,7,10,2.0,1,1,1,3
4,FOODS_1_001,CA_1,2011,8,14,2.0,1,0,1,4


## Lag Features

Historical demand is one of the strongest predictors of future demand.

Lag features capture sales observed in previous weeks while ensuring that only past information is available to the model. This prevents data leakage because future observations are never used as predictors.

The following lag intervals are generated:

Lag 1 week
Lag 2 weeks
Lag 4 weeks
Lag 8 weeks


In [11]:
# ============================================================
# Lag Features
# ============================================================

weekly_df["lag_1"] = (
    weekly_df
    .groupby(["item_id", "store_id"])["units_sold"]
    .shift(1)
)

weekly_df["lag_2"] = (
    weekly_df
    .groupby(["item_id", "store_id"])["units_sold"]
    .shift(2)
)

weekly_df["lag_4"] = (
    weekly_df
    .groupby(["item_id", "store_id"])["units_sold"]
    .shift(4)
)

weekly_df["lag_8"] = (
    weekly_df
    .groupby(["item_id", "store_id"])["units_sold"]
    .shift(8)
)

weekly_df.head()

,item_id,store_id,year,week_of_year,units_sold,sell_price,has_event,has_snap,is_weekend,time_idx,lag_1,lag_2,lag_4,lag_8
0,FOODS_1_001,CA_1,2011,4,3,2.0,0,0,1,0,NaN,NaN,NaN,NaN
1,FOODS_1_001,CA_1,2011,5,9,2.0,1,1,1,1,3.0,NaN,NaN,NaN
2,FOODS_1_001,CA_1,2011,6,7,2.0,0,1,1,2,9.0,3.0,NaN,NaN
3,FOODS_1_001,CA_1,2011,7,10,2.0,1,1,1,3,7.0,9.0,NaN,NaN
4,FOODS_1_001,CA_1,2011,8,14,2.0,1,0,1,4,10.0,7.0,3.0,NaN


## Rolling Statistics

Rolling statistics summarize recent demand behaviour over multiple weeks.

Unlike simple lag variables, rolling averages smooth short-term fluctuations and provide information about underlying demand trends. Rolling statistics are computed only from historical observations by shifting the target variable before applying the rolling window.

In [12]:
# ============================================================
# Rolling Mean Features
# ============================================================

weekly_df["rolling_mean_4"] = (
    weekly_df
    .groupby(["item_id", "store_id"])["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(4).mean()
    )
)

weekly_df["rolling_mean_8"] = (
    weekly_df
    .groupby(["item_id", "store_id"])["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(8).mean()
    )
)

In [13]:
# ============================================================
# Rolling Standard Deviation
# ============================================================

weekly_df["rolling_std_4"] = (
    weekly_df
    .groupby(["item_id", "store_id"])["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(4).std()
    )
)

weekly_df["rolling_std_8"] = (
    weekly_df
    .groupby(["item_id", "store_id"])["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(8).std()
    )
)

In [14]:
lag_columns = [
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "rolling_mean_4",
    "rolling_std_4"
]

weekly_df[lag_columns] = weekly_df[lag_columns].fillna(0)

## Time-Based Train-Test Split

Random train-test splitting is inappropriate for forecasting because it mixes future observations into the training data.

Instead, the dataset is split chronologically so that the model is trained on historical observations and evaluated on future weeks.

This mirrors real-world forecasting conditions and prevents information leakage.

In [15]:
split = int(len(weekly_df) * 0.8)

train = weekly_df.iloc[:split].copy()

test = weekly_df.iloc[split:].copy()

print(train.shape)

print(test.shape)

(8122536, 18)
(2030634, 18)


## Seasonal Naive Baseline
## Objective

Before training a machine learning model, a simple forecasting baseline is established.

The seasonal-naive approach assumes that demand in the current week will be similar to demand observed in the corresponding week of the previous seasonal cycle.

This benchmark ensures that the forecasting model provides measurable value beyond a straightforward historical heuristic.

The baseline forecast serves as the reference model against which all machine learning models will be evaluated using WAPE.

In [16]:
# ============================================================
# Seasonal Naive Baseline
# ============================================================

test = test.copy()

test["baseline_forecast"] = test["lag_1"]

test[
    [
        "units_sold",
        "baseline_forecast"
    ]
].head()

,units_sold,baseline_forecast
8122536,0,0.0
8122537,0,0.0
8122538,0,0.0
8122539,0,0.0
8122540,0,0.0


## Weighted Absolute Percentage Error (WAPE)
## Objective

Forecast accuracy is evaluated using Weighted Absolute Percentage Error (WAPE), the metric specified by the Project FORESIGHT client.

Unlike MAPE, WAPE remains stable when actual demand contains zeros and provides a meaningful measure of aggregate forecasting accuracy across all SKUs.

Lower WAPE values indicate better forecasting performance.

In [17]:
# ============================================================
# WAPE Function
# ============================================================

def wape(actual, predicted):
    return (
        abs(actual - predicted).sum()
        /
        abs(actual).sum()
    )

baseline_wape = wape(
    test["units_sold"],
    test["baseline_forecast"]
)

print(f"Baseline WAPE : {baseline_wape:.4f}")

Baseline WAPE : 0.7141


# Random Forest Demand Forecast Model

## Objective

A Random Forest Regressor is trained to forecast weekly SKU demand.

Random Forest was selected because it:

- Captures non-linear relationships
- Handles mixed numerical features
- Is robust to noisy retail demand
- Requires minimal feature scaling
- Provides feature importance for interpretation

The model is trained only on historical observations, ensuring that future information never enters the learning process.

In [18]:
# ============================================================
# Import Model
# ============================================================

from sklearn.ensemble import RandomForestRegressor

In [19]:
# ============================================================
# Select Model Features
# ============================================================

feature_columns = [

    "sell_price",
    "has_event",
    "has_snap",
    "is_weekend",

    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",

    "rolling_mean_4",
    "rolling_std_4"

]

target = "units_sold"

In [20]:
# ============================================================
# Prepare Training Data
# ============================================================

X_train = train[feature_columns]

y_train = train[target]

X_test = test[feature_columns]

y_test = test[target]

In [21]:
# ============================================================
# Train Random Forest
# ============================================================

rf = RandomForestRegressor(

    n_estimators=100,

    max_depth=15,

    random_state=42,

    n_jobs=-1

)

rf.fit(
    X_train,
    y_train
)

print("Random Forest training completed.")

Random Forest training completed.


# 4.11 Forecast Generation

The trained Random Forest model is used to generate weekly demand forecasts for the unseen test dataset.

These predictions will be compared against the seasonal-naive baseline using WAPE.

In [22]:
# ============================================================
# Predict
# ============================================================

test["rf_forecast"] = rf.predict(X_test)

test[
    [
        "units_sold",
        "baseline_forecast",
        "rf_forecast"
    ]
].head()

,units_sold,baseline_forecast,rf_forecast
8122536,0,0.0,0.0
8122537,0,0.0,0.0
8122538,0,0.0,0.0
8122539,0,0.0,0.0
8122540,0,0.0,0.0


In [23]:
# ============================================================
# Random Forest WAPE
# ============================================================

rf_wape = wape(
    test["units_sold"],
    test["rf_forecast"]
)

print(f"Baseline WAPE      : {baseline_wape:.4f}")

print(f"Random Forest WAPE : {rf_wape:.4f}")

Baseline WAPE      : 0.7141
Random Forest WAPE : 0.5905


# 4.13 Model Comparison

The forecasting model is accepted only if it performs better than the seasonal-naive benchmark.

The comparison below identifies the best-performing model based on WAPE.

In [24]:
# ============================================================
# Compare Models
# ============================================================

if rf_wape < baseline_wape:

    print("Random Forest outperformed the seasonal-naive baseline.")

else:

    print("Seasonal-naive baseline performed better.")

Random Forest outperformed the seasonal-naive baseline.


#  Feature Importance

Random Forest provides feature importance scores that indicate which variables contribute most to forecasting demand.

These insights improve model interpretability and help identify the primary demand drivers.

In [25]:
# ============================================================
# Feature Importance
# ============================================================

importance = (

    pd.DataFrame({

        "Feature": feature_columns,

        "Importance": rf.feature_importances_

    })

    .sort_values(

        "Importance",

        ascending=False

    )

)

importance

,Feature,Importance
8,rolling_mean_4,0.705071
4,lag_1,0.134529
3,is_weekend,0.064108
9,rolling_std_4,0.031139
7,lag_8,0.016711
5,lag_2,0.015034
0,sell_price,0.012622
6,lag_4,0.012514
1,has_event,0.005211
2,has_snap,0.003061


#  Save Forecast Results

The final weekly demand forecasts are stored for downstream risk scoring, dashboard visualization, and deployment.

These predictions will serve as the input for stockout and overstock risk assessment in the next notebook.

In [26]:
# ============================================================
# Save Forecast Results
# ============================================================

forecast_columns = [

    "item_id",
    "store_id",
    "year",
    "week_of_year",

    "units_sold",

    "baseline_forecast",

    "rf_forecast"

]

forecast_results = test[forecast_columns]

forecast_results.to_parquet(
    PROCESSED_DATA_DIR / "weekly_forecasts.parquet",
    index=False
)

print("Forecast results saved successfully.")

Forecast results saved successfully.


# Save Trained Model

The trained Random Forest model is serialized for reuse in the dashboard and deployed scoring service.

Persisting the model ensures forecasts can be generated without retraining.

In [30]:
import joblib

MODEL_DIR = PROCESSED_DATA_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    rf,
    MODEL_DIR / "random_forest_forecaster.pkl"
)

print("✅ Model saved successfully.")

✅ Model saved successfully.


In [31]:
# ============================================================
# Notebook 04 Validation
# ============================================================

print("=" * 50)
print("PROJECT FORESIGHT - NOTEBOOK 04 VALIDATION")
print("=" * 50)

print(f"Training Records      : {len(train):,}")
print(f"Testing Records       : {len(test):,}")

print(f"\nBaseline WAPE         : {baseline_wape:.4f}")
print(f"Random Forest WAPE    : {rf_wape:.4f}")

print("\nTop 10 Important Features")

display(importance.head(10))

print("\nForecast File Exists:",
      (PROCESSED_DATA_DIR / "weekly_forecasts.parquet").exists())

print("Model File Exists:",
      (MODEL_DIR / "random_forest_forecaster.pkl").exists())

print("\nNotebook 04 Completed Successfully!")

PROJECT FORESIGHT - NOTEBOOK 04 VALIDATION
Training Records      : 8,122,536
Testing Records       : 2,030,634

Baseline WAPE         : 0.7141
Random Forest WAPE    : 0.5905

Top 10 Important Features


,Feature,Importance
8,rolling_mean_4,0.705071
4,lag_1,0.134529
3,is_weekend,0.064108
9,rolling_std_4,0.031139
7,lag_8,0.016711
5,lag_2,0.015034
0,sell_price,0.012622
6,lag_4,0.012514
1,has_event,0.005211
2,has_snap,0.003061



Forecast File Exists: True
Model File Exists: True

Notebook 04 Completed Successfully!


# Notebook Conclusion

This notebook successfully developed and evaluated the demand forecasting model for Project FORESIGHT.

## Deliverables Completed

- Weekly SKU-level demand forecasting
- Seasonal-naive baseline model
- Time-series feature engineering
- Lag and rolling statistics
- Chronological train-test split
- Random Forest forecasting model
- Forecast evaluation using WAPE
- Model comparison against baseline
- Feature importance analysis
- Saved forecast outputs
- Saved trained model

The resulting forecasts will be used in the next stage to estimate stockout and overstock risk and generate inventory recommendations.